# Session 2 - Joins and Aggregations

Welcome. In this session we will learn:

- how to **combine tables** with JOIN
- the difference between **INNER** and **LEFT** joins
- how to **summarize** rows with `GROUP BY` and `HAVING`

Sample tables are in database **`training`** on **your** MySQL. Use names like `training.customers`.


---
# Chapter 1 - Why JOINs?

Customers live in one table. Orders live in another.

We do **not** copy the full company name into every order.

Instead:

- Keep the company once in `customers`
- Store only `customer_id` on each order
- Use a **JOIN** when we need both sides of the story

Then run the first query.


**JOIN types in one glance**

| Join | Keeps | Idea |
|------|-------|------|
| **INNER JOIN** | only matching rows | customer **and** order both exist |
| **LEFT JOIN** | all left rows | all customers; orders when present |
| Unmatched rows | gaps on either side | find missing links / orphans |

In this session we practice INNER and LEFT joins on `training` tables.


### Browse orders (notice customer_id)

Run this. Notice `customer_id` on each order.


In [1]:
SELECT order_id, customer_id, order_date, status
FROM training.orders
ORDER BY order_id
LIMIT 8;


+----------+-------------+------------+------------+
| order_id | customer_id | order_date | status     |
+----------+-------------+------------+------------+
|        1 |           1 | 2023-01-02 | processing |
|        2 |           2 | 2023-01-03 | shipped    |
|        3 |           3 | 2023-01-04 | cancelled  |
|        4 |           4 | 2023-01-05 | delivered  |
|        5 |           5 | 2023-01-06 | pending    |
|        6 |           6 | 2023-01-07 | processing |
|        7 |           7 | 2023-01-08 | shipped    |
|        8 |           8 | 2023-01-09 | cancelled  |
+----------+-------------+------------+------------+


---
# Chapter 2 - INNER JOIN

**INNER JOIN** keeps a row only when **both** sides match.

Story: "Show company name next to each order."

Pattern:

```sql
FROM training.orders o
JOIN training.customers c ON c.customer_id = o.customer_id
```

`JOIN` without a word means **INNER JOIN**.


### Orders with company names


In [2]:
SELECT o.order_id, o.order_date, o.status,
       c.company_name, c.city, c.country
FROM training.orders o
JOIN training.customers c ON c.customer_id = o.customer_id
ORDER BY o.order_id
LIMIT 15;


+----------+------------+------------+--------------------------------+-----------+-----------+
| order_id | order_date | status     | company_name                   | city      | country   |
+----------+------------+------------+--------------------------------+-----------+-----------+
|        1 | 2023-01-02 | processing | Nimbus Softwares Pvt Ltd       | Mumbai    | India     |
|        2 | 2023-01-03 | shipped    | Helios FinServe Pvt Ltd        | Delhi     | India     |
|        3 | 2023-01-04 | cancelled  | Apex Technologies Pvt Ltd      | Bengaluru | India     |
|        4 | 2023-01-05 | delivered  | Bright Networks Pvt Ltd        | Hyderabad | India     |
|        5 | 2023-01-06 | pending    | Summit Manufacturing Pvt Ltd   | Chennai   | India     |
|        6 | 2023-01-07 | processing | Orbit Solutions Pvt Ltd        | Pune      | India     |
|        7 | 2023-01-08 | shipped    | Vertex Analytics Pvt Ltd       | Kolkata   | India     |
|        8 | 2023-01-09 | cancelled  | H

### Order lines with product names

Three tables: orders -> order_items -> products.


In [3]:
SELECT oi.order_id, p.product_name, oi.quantity, oi.unit_price,
       round(oi.quantity * oi.unit_price * (1 - oi.discount), 2) AS line_total
FROM training.order_items oi
JOIN training.products p ON p.product_id = oi.product_id
WHERE oi.order_id BETWEEN 100 AND 110
ORDER BY oi.order_id, oi.order_item_id;


+----------+-----------------------------------------+----------+------------+------------+
| order_id | product_name                            | quantity | unit_price | line_total |
+----------+-----------------------------------------+----------+------------+------------+
|      100 | Corsair Evolve2 65 / SKU-0100           |        5 |   11836.47 |   59182.35 |
|      100 | Apple 980 PRO 1TB / SKU-1100            |        5 |    2958.44 |   14792.20 |
|      100 | AWS Credits USD 500 / SKU-0600          |        5 |   15824.70 |   71211.15 |
|      101 | Cisco C920e / SKU-0101                  |        6 |   13452.68 |   80716.08 |
|      101 | Samsung Archer AX55 / SKU-1101          |        6 |   18839.00 |  113034.00 |
|      101 | HP EliteBook 840 / SKU-0601             |        6 |   96305.72 |  577834.32 |
|      102 | Asus Surface Go 4 / SKU-0102            |        7 |   20851.40 |  145959.80 |
|      102 | Lenovo Thunderbolt Dock / SKU-1102      |        7 |    2316.63 |  

<div style="background-color:#FFF4CC; border:2px solid #E6A800; border-radius:8px; padding:16px 18px; margin:16px 0;">
<h2 style="margin:0 0 10px 0; color:#92400E;">Quiz</h2>
<p style="margin:0 0 10px 0; font-size:1.05em; font-weight:600;">SQL - Session 2 - INNER JOIN</p>
<p style="margin:0 0 10px 0;"><a href="https://vmreact.eastus2.cloudapp.azure.com:18094/a/sql-s02-inner-join-01">https://vmreact.eastus2.cloudapp.azure.com:18094/a/sql-s02-inner-join-01</a></p>
<p style="margin:0; color:#444;">Use the login ID your trainer provided. Every student must attempt this quiz.</p>
</div>


---
# Chapter 3 - LEFT JOIN

**LEFT JOIN** keeps **every** row from the left table.

If the right side has no match, those columns are `NULL`.

Story: "List all customers, and show how many orders each has (zero is ok)."


### Customers and order counts (including zero)


In [4]:
SELECT c.customer_id, c.company_name,
       COUNT(o.order_id) AS order_count
FROM training.customers c
LEFT JOIN training.orders o ON o.customer_id = c.customer_id
GROUP BY c.customer_id, c.company_name
ORDER BY order_count ASC, c.customer_id
LIMIT 20;


+-------------+----------------------------------------+-------------+
| customer_id | company_name                           | order_count |
+-------------+----------------------------------------+-------------+
|          50 | Pioneer Cloud Services Pvt Ltd         |           0 |
|         100 | Silverline Trading Co LLC              |           0 |
|         150 | Northstar Cloud Services Pvt Ltd       |           0 |
|         200 | Ironclad Trading Co LLC                |           0 |
|         250 | Pioneer Cloud Services Pvt Ltd         |           0 |
|         300 | Silverline Trading Co LLC              |           0 |
|         350 | Northstar Cloud Services Pvt Ltd       |           0 |
|         400 | Ironclad Trading Co LLC                |           0 |
|         450 | Pioneer Cloud Services Pvt Ltd         |           0 |
|         500 | Silverline Trading Co LLC              |           0 |
|         550 | Northstar Cloud Services Pvt Ltd       |           0 |
|     

### Orders with optional payment

Some orders have no payment row. LEFT JOIN shows them.


In [5]:
SELECT o.order_id, o.status, o.order_date,
       p.payment_id, p.amount, p.status AS payment_status
FROM training.orders o
LEFT JOIN training.payments p ON p.order_id = o.order_id
WHERE o.order_id BETWEEN 200 AND 220
ORDER BY o.order_id;


+----------+------------+------------+------------+------------+----------------+
| order_id | status     | order_date | payment_id | amount     | payment_status |
+----------+------------+------------+------------+------------+----------------+
|      200 | pending    | 2023-07-20 |        134 |   21562.79 | authorized     |
|      201 | processing | 2023-07-21 |       NULL |       NULL | NULL           |
|      202 | shipped    | 2023-07-22 |        135 |   91830.49 | failed         |
|      203 | cancelled  | 2023-07-23 |        136 |  424560.07 | refunded       |
|      204 | delivered  | 2023-07-24 |       NULL |       NULL | NULL           |
|      205 | pending    | 2023-07-25 |        137 |  639998.06 | authorized     |
|      206 | processing | 2023-07-26 |        138 |  987812.80 | captured       |
|      207 | shipped    | 2023-07-27 |       NULL |       NULL | NULL           |
|      208 | cancelled  | 2023-07-28 |        139 |   83914.51 | refunded       |
|      209 | del

---
# Chapter 4 - Finding rows with no match on the other side

Sometimes you need rows that **do not** join cleanly:

- customers with **no** orders
- orders with **no** matching customer (orphans)


### Unmatched customers and orphan orders


In [6]:
SELECT c.customer_id, c.company_name, o.order_id, o.status
FROM training.customers c
LEFT JOIN training.orders o ON o.customer_id = c.customer_id
WHERE o.order_id IS NULL
UNION ALL
SELECT c.customer_id, c.company_name, o.order_id, o.status
FROM training.orders o
LEFT JOIN training.customers c ON c.customer_id = o.customer_id
WHERE c.customer_id IS NULL
ORDER BY order_id, customer_id
LIMIT 25;


+-------------+----------------------------------------+----------+--------+
| customer_id | company_name                           | order_id | status |
+-------------+----------------------------------------+----------+--------+
|          50 | Pioneer Cloud Services Pvt Ltd         |     NULL | NULL   |
|         100 | Silverline Trading Co LLC              |     NULL | NULL   |
|         150 | Northstar Cloud Services Pvt Ltd       |     NULL | NULL   |
|         200 | Ironclad Trading Co LLC                |     NULL | NULL   |
|         250 | Pioneer Cloud Services Pvt Ltd         |     NULL | NULL   |
|         300 | Silverline Trading Co LLC              |     NULL | NULL   |
|         350 | Northstar Cloud Services Pvt Ltd       |     NULL | NULL   |
|         400 | Ironclad Trading Co LLC                |     NULL | NULL   |
|         450 | Pioneer Cloud Services Pvt Ltd         |     NULL | NULL   |
|         500 | Silverline Trading Co LLC              |     NULL | NULL   |

<div style="background-color:#FFF4CC; border:2px solid #E6A800; border-radius:8px; padding:16px 18px; margin:16px 0;">
<h2 style="margin:0 0 10px 0; color:#92400E;">Quiz</h2>
<p style="margin:0 0 10px 0; font-size:1.05em; font-weight:600;">SQL - Session 2 - LEFT JOIN & unmatched rows</p>
<p style="margin:0 0 10px 0;"><a href="https://vmreact.eastus2.cloudapp.azure.com:18094/a/sql-s02-left-join-01">https://vmreact.eastus2.cloudapp.azure.com:18094/a/sql-s02-left-join-01</a></p>
<p style="margin:0; color:#444;">Use the login ID your trainer provided. Every student must attempt this quiz.</p>
</div>


---
# Chapter 5 - GROUP BY and aggregates

A JOIN returns many detail rows. Aggregates turn them into **summaries**.

Common aggregates:

- `COUNT(*)` - how many rows
- `COUNT(column)` - how many non-null values
- `SUM(column)` - total
- `AVG(column)` - average
- `MIN` / `MAX` - extremes

Rule: every selected column must be either aggregated **or** listed in `GROUP BY`.


**GROUP BY then HAVING**

| Step | What happens |
|------|----------------|
| 1. Rows | Many order rows |
| 2. `GROUP BY` | Bundle rows (for example by country) |
| 3. Aggregate | `COUNT` / `SUM` / `AVG` per group |
| 4. `HAVING` | Keep only groups that pass a filter |

`WHERE` filters rows **before** grouping. `HAVING` filters groups **after** aggregating.


### Orders per country


In [7]:
SELECT c.country,
       COUNT(*) AS order_count,
       round(AVG(o.freight), 2) AS avg_freight
FROM training.orders o
JOIN training.customers c ON c.customer_id = o.customer_id
GROUP BY c.country
ORDER BY order_count DESC;


+-----------+-------------+-------------+
| country   | order_count | avg_freight |
+-----------+-------------+-------------+
| India     |       19182 |      290.22 |
| USA       |        5873 |      290.79 |
| UAE       |        3916 |      290.77 |
| Germany   |        3914 |      288.95 |
| UK        |        3523 |      285.45 |
| Singapore |        1958 |      290.73 |
+-----------+-------------+-------------+


### Revenue by category (line totals)


In [8]:
SELECT cat.category_name,
       COUNT(*) AS line_count,
       round(SUM(oi.quantity * oi.unit_price * (1 - oi.discount)), 2) AS revenue
FROM training.order_items oi
JOIN training.products p ON p.product_id = oi.product_id
JOIN training.categories cat ON cat.category_id = p.category_id
GROUP BY cat.category_name
ORDER BY revenue DESC;


+---------------+------------+----------------+
| category_name | line_count | revenue        |
+---------------+------------+----------------+
| Servers       |       8000 | 16378002676.91 |
| Laptops       |       8000 |  3299810218.58 |
| Cloud Credits |       8000 |  2005086032.04 |
| Phones        |       8000 |  1923453187.23 |
| Tablets       |       8000 |  1419653994.32 |
| Printers      |       8000 |   994533232.93 |
| Networking    |       8000 |   849588513.22 |
| Software      |       8000 |   701543956.26 |
| Monitors      |       8000 |   700371287.93 |
| Cameras       |       8000 |   444689923.83 |
| Storage       |       8000 |   425694502.43 |
| Audio         |       8000 |   258798026.16 |
| Keyboards     |       8000 |   113197554.18 |
| Mice          |       8000 |    84954975.91 |
| Accessories   |       8000 |    73497200.70 |
+---------------+------------+----------------+


<div style="background-color:#FFF4CC; border:2px solid #E6A800; border-radius:8px; padding:16px 18px; margin:16px 0;">
<h2 style="margin:0 0 10px 0; color:#92400E;">Quiz</h2>
<p style="margin:0 0 10px 0; font-size:1.05em; font-weight:600;">SQL - Session 2 - GROUP BY & aggregates</p>
<p style="margin:0 0 10px 0;"><a href="https://vmreact.eastus2.cloudapp.azure.com:18094/a/sql-s02-groupby-01">https://vmreact.eastus2.cloudapp.azure.com:18094/a/sql-s02-groupby-01</a></p>
<p style="margin:0; color:#444;">Use the login ID your trainer provided. Every student must attempt this quiz.</p>
</div>


---
# Chapter 6 - HAVING

`WHERE` filters **rows** before grouping.

`HAVING` filters **groups** after aggregation.

Story: "Only countries with more than 3000 orders."


### Busy countries only


In [9]:
SELECT c.country,
       COUNT(*) AS order_count
FROM training.orders o
JOIN training.customers c ON c.customer_id = o.customer_id
GROUP BY c.country
HAVING COUNT(*) > 3000
ORDER BY order_count DESC;


+---------+-------------+
| country | order_count |
+---------+-------------+
| India   |       19182 |
| USA     |        5873 |
| UAE     |        3916 |
| Germany |        3914 |
| UK      |        3523 |
+---------+-------------+


<div style="background-color:#FFF4CC; border:2px solid #E6A800; border-radius:8px; padding:16px 18px; margin:16px 0;">
<h2 style="margin:0 0 10px 0; color:#92400E;">Quiz</h2>
<p style="margin:0 0 10px 0; font-size:1.05em; font-weight:600;">SQL - Session 2 - HAVING</p>
<p style="margin:0 0 10px 0;"><a href="https://vmreact.eastus2.cloudapp.azure.com:18094/a/sql-s02-having-01">https://vmreact.eastus2.cloudapp.azure.com:18094/a/sql-s02-having-01</a></p>
<p style="margin:0; color:#444;">Use the login ID your trainer provided. Every student must attempt this quiz.</p>
</div>
